# Bonus: Architektura pamięci - gdzie trzymać dane agenta?

## Cel: Dobranie technologii bazodanowej, która przetrwa zderzenie z produkcją i oszczędzi ci miesięcy debugowania

### Czego się nauczysz:
1. Dlaczego większość systemów AI wykłada się na prostych pytaniach o fakty?
2. Kiedy baza wektorowa zwróci Ci halucynacje zamiast twardych danych?
3. Na czym polega pułapka baz grafowych i dlaczego Neo4j to często overkill?
4. Jak zmapować pamięć epizodyczną, semantyczną i roboczą na konkretne silniki?
5. Dlaczego twój kod logiczny nigdy nie powinien wiedzieć, jakiej bazy używa?
6. Czym w praktyce różni się praca z Redisem, Chromą i Qdrantem?
7. Jak tanio prototypować lokalnie i bezboleśnie przejść na skalę produkcyjną?


Zacznijmy, tak jak zwykle, od brutalnej prawdy, której nie usłyszysz na marketingowych konferencjach AI.

**Większość projektów AI nie upada z powodu złego modelu LLM. Upada z powodu źle zaprojektowanej warstwy danych.**

W świecie Software 3.0 panuje obecnie niebezpieczny wirus: "Vector Hype". Inżynierowie masowo wrzucają absolutnie wszystko do bazy wektorowej – od dokumentacji technicznej, przez historię czatu, aż po cenniki produktów. A potem następuje wielkie zdziwienie, gdy system na pytanie "Jaka jest cena produktu X?" odpowiada halucynacją lub artykułem o strategii cenowej, zamiast podać konkretną kwotę.

Dlaczego? Bo wektory służą do szukania *podobieństwa*, a nie *faktów*.

Twoim celem jako inżyniera nie jest użycie najmodniejszej bazy danych (korzystając z podejścia Linkedin Driven Development). Twoim celem jest zbudowanie systemu, który działa na produkcji. Dlatego dzisiaj dostaniesz ode mnie nie tutorial do jednej bazy, ale matrycę decyzyjną.


## Wielkie <s>kłamstwo</s> iluzja baz wektorowych 🤥

Spójrzmy na to na chłodno, bez emocji. Marketing baz wektorowych sprzedaje nam wizję: 'wrzuć dane, a embeddingi załatwią resztę'. W zderzeniu z produkcją to jednak obietnica bez pokrycia.

Weterani branży ostrzegają wprost: **Embeddingi są świetne do znajdowania "mniej więcej", ale beznadziejne do znajdowania "dokładnie to".**

### Problem: Iluzja semantyki

Kiedy pytasz bazę wektorową o "najtańszy produkt", ona nie szuka w kolumnie `price` wartości minimalnej. Ona szuka tekstu, który semantycznie przypomina słowo "najtańszy". To fundamentalna różnica.

| Cecha | Baza relacyjna (SQL) | Baza wektorowa |
| :--- | :--- | :--- |
| **Pytanie** | "Daj mi rekordy gdzie user_id=123" | "Daj mi coś podobnego do tego tekstu" |
| **Gwarancja** | 100% precyzji (ACID) | Przybliżenie (Approximate Nearest Neighbor) |
| **Debugowanie** | Przejrzyste (widzisz dane) | "Czarna skrzynka" (dlaczego ten wektor jest blisko?) |
| **Zastosowanie** | Fakty, Logi, Konfiguracja | Wyszukiwanie rozmyte, Clustering |

### ☝️ Kiedy wektory kłamią?
Wyobraź sobie, że budujesz system dla e-commerce.
1. Użytkownik pyta: *"Ile kosztuje iPhone 15?"*
2. **Baza wektorowa** zwraca: Artykuł z bloga pt. "Dlaczego iPhone 15 jest wart swojej ceny".
3. **Baza SQL** zwraca: `5999.00 PLN`.

Rozumiesz różnicę? Baza wektorowa zwróciła coś *semantycznie powiązanego*, ale *bezużytecznego* w kontekście precyzyjnego pytania o fakt.

> **Złota zasada inżyniera:** Jeśli potrzebujesz 100% pewności (np. ID użytkownika, cena, data, stan zamówienia), używasz SQL. Jeśli akceptujesz "szum" i 80% trafności (np. "znajdź podobne artykuły"), używasz wektorów.

### ⚠️ Ważne zastrzeżenie: To nie jest nagonka na technologię

Żebyśmy się dobrze zrozumieli: **Bazy wektorowe nie są ani złe, ani dobre. Są po prostu narzędziem.** Krytykuję tutaj podejście, a nie technologię.

Wiele osób wpada w pułapkę myślenia, że wektory to "magia" – wrzucasz tam cokolwiek i nagle system "rozumie". **Nie, nie rozumie!**


### Iluzja "automatycznej semantyki"

```
❌ Co ludzie zwykle myślą:
"Wrzucę dokumenty do bazy wektorowej i magicznie będę mógł 
wyszukiwać 'semantycznie'"

✅ Co się naprawdę dzieje:
- Embedding model ma swoje ograniczenia i biasy
- "Podobieństwo wektorowe" ≠ "przydatność dla mojego use case"
- Garbage in = Garbage out (ale teraz z kosinusowym podobieństwem!)
```


Empirycznie powiem Ci coś, co może zabrzmieć kontraintuicyjnie: zaprojektowanie działającego systemu opartego na bazie wektorowej wymaga ZNACZNIE WIĘCEJ wysiłku inżynierskiego, niż przy klasycznej bazie SQL.

Dlaczego?
1.  **Brak gwarancji:** Baza wektorowa nie daje Ci gwarancji jednoznaczności.
2.  **Odporność na błędy:** Musisz zaprojektować system tak, aby był odporny na "halucynacje retrievalu" (sytuacje, gdy baza zwróci coś podobnego matematycznie, ale bezsensownego logicznie).
3.  **Konieczność planowania:** Mimo że bazy te są często „schemaless”, musisz precyzyjnie zaplanować strukturę metadanych, strategię dzielenia tekstu oraz dobór modelu embedującego. To cały szereg decyzji podejmowanych pod spodem.

Są sytuacje – np. gdy masz miliony nieustrukturyzowanych dokumentów PDF – gdzie nie masz innego wyboru i baza wektorowa jest jedynym ratunkiem. Ale wtedy musisz traktować ją tak samo jak LLM-a: jako komponent niedeterministyczny. To bardzo ważne założenie, jeśli chcesz budować stabilne rozwiązania.

Twoim zadaniem jest zbudowanie systemu dookoła niej w taki sposób, aby jej słabe strony (brak precyzji) nie przewróciły całego projektu. Domyślnie – jeśli tego nie zaprojektujesz – system się wyłoży.


### Kiedy baza wektorowa ma sens?

Na podstawie doświadczeń powiem tak:

```
✅ DOBRE zastosowania baz wektorowych:

1. Wyszukiwanie "podobnych" gdy nie wiesz dokładnie czego szukasz
   - "Znajdź produkty podobne do tego"
   - "Pokaż artykuły o zbliżonej tematyce"

2. Jako JEDEN z sygnałów w hybrydowym wyszukiwaniu
   - Wektorowe + keyword + filtry metadata
   - ale nie samo wektorowe!

3. Deduplikacja i wykrywanie near-duplicates
   - To jedno z najbardziej niedocenianych zastosowań

4. Clustering eksploracyjny
   - "Pokaż mi grupy podobnych dokumentów"
```

```
❌ ZŁE zastosowania (gdzie ludzie się palą):

1. "Semantic search" jako jedyny mechanizm wyszukiwania
   - User pyta: "Jaka jest cena produktu X?"
   - System zwraca: artykuł o strategiach cenowych
   
2. RAG bez pre-processingu danych
   - Wrzucenie surowych PDF-ów
   - "Chunking" bez zrozumienia struktury dokumentu
   
3. Zastępowanie relacyjnej bazy tam, gdzie potrzebujesz ACID
   - "Zapiszmy sesje użytkowników jako embeddingi"
   - NO. NO. NO :)
```


## Bazy grafowe – potęga relacji (i jeszcze większa pułapka)

Zanim przejdziemy do ostrzeżeń, wyjaśnijmy sobie jedną rzecz, bo nie każdy miał okazję pracować z [Neo4j](https://neo4j.com) czy [falkordb](https://www.falkordb.com).


Opowiem Ci autentyczną historię. Zgłosiła się do nas pewna firma z klasycznym problemem: "Rozwiązanie nie działa, a czas nas goni". Męczyli się z tematem prawie rok samodzielnie, został im miesiąc do deadline'u i byli pod ścianą.

Patrząc na objawy, zadałem moje standardowe pytanie startowe, spodziewając się klasycznego błędu nowicjusza:
– Prawdopodobnie wrzuciliście wszystko jak leci do bazy wektorowej?

Na co usłyszałem odpowiedź pełną inżynierskiej dumy:
– **Nie, skądże! My mamy bazy grafowe.**


Moja reakcja? Zapewniam Cię, że ostatnią rzeczą, na którą miałem ochotę, był śmiech. W profesjonalnej inżynierii nie ma miejsca na wyśmiewanie klienta, który stoi nad przepaścią. W środku poczułem jednak głęboki dysonans poznawczy 🤦‍. Zobaczyłem zespół, który tonie, a na ratunek rzuca sobie... kowadło.

Okazało się, że w akcie desperacji połączyli bazę wektorową z bazą grafową. Stworzyli system o maksymalnej możliwej złożoności, wierząc, że technologia magicznie rozwiąże braki w logice biznesowej.

Dlaczego to tak boli? Bo jeśli masz problem z dowiezieniem podstawowej funkcjonalności (MVP), to dołożenie do tego dwóch najbardziej wymagających w utrzymaniu technologii jest jak gaszenie pożaru benzyną. Zamiast uprościć problem, skomplikowali go do poziomu, w którym nikt już nie rozumiał, co się dzieje "pod maską".

Epilog tej historii? Próbowałem im to wytłumaczyć. Sugerowałem krok w tył i uproszczenie architektury. Nie uwierzyli – iluzja, że "zaawansowane AI wymaga skomplikowanej bazy", była zbyt silna. Niestety, tego projektu nie udało się uratować. Nie pozwól, by Twój skończył tak samo.



To prowadzi nas do kluczowego wniosku tej lekcji: **Technologia nie jest rozwiązaniem Twojego problemu. Często jest jego źródłem.**


### Czym w ogóle jest baza grafowa?
Najprościej: wyobraź sobie różnicę między **arkuszem Excela** a **tablicą detektywa** z filmu kryminalnego.

- Baza relacyjna (SQL/Excel): To zbiór oddzielnych tabel. Aby połączyć dane (np. klienta z zamówieniem), musisz wykonać operację `JOIN`. Dla komputera to kosztowne przeszukiwanie i dopasowywanie kluczy. Relacja jest tutaj czymś wtórnym – powstaje dopiero w momencie zapytania.
- Baza grafowa (Neo4j): To tablica ze zdjęciami (Węzły/Nodes) połączonymi czerwonymi sznurkami (krawędzie/edges). Kluczowa różnica? Relacja (sznurek) jest tu tak samo ważna jak dane. Jest fizycznie zapisana na dysku. Przejście od "Jana" do "Firmy X" nie wymaga przeszukiwania tabeli – to po prostu przeskok po wskaźniku w pamięci (index-free adjacency).

![](../images/graph_example.jpg)

Brzmi jak idealne rozwiązanie do mapowania złożonego świata? Tak. I właśnie dlatego jest to tak niebezpieczne.

### Dlaczego grafy są "sexy", ale ryzykowne?

Problem polega na tym, że grafy kuszą, by wrzucić do nich wszystko. A to błąd.

**Michael Hunger** (wieloletni Developer Relations Lead w Neo4j) – człowiek, który żyje z promowania grafów – sam ostrzega:

> "Największy błąd, jaki widzę: ludzie modelują w grafie to, co powinno być tabelą. Graf ma sens, gdy RELACJE są obywatelami pierwszej klasy twojego modelu. Jeśli twoje zapytania to głównie 'daj mi X gdzie Y=Z', zostań przy SQL."

### Praktyczna heurystyka (test detektywa)

Zanim zainstalujesz bazę grafową, zrób ten szybki test. Spójrz na swoje typowe zapytania:

- A) "Znajdź wszystkich klientów z Warszawy" → **SQL**. To proste filtrowanie atrybutów. Nawet nie myśl o grafie.
- B) "Pokaż wszystkie zamówienia z ostatniego miesiąca" → 🛑 **SQL**. Proszę, błagam, zostań przy relacyjnej bazie. To klasyczna agregacja.
- C) "Znajdź najkrótszą ścieżkę wpływu między osobą A a decyzją X" → ✅ Graf. Tu liczy się ścieżka i relacja, a nie same dane.
- D) "Kto zna kogoś, kto zna kogoś, kto pracował nad projektem Y?" → ✅ Graf. To jest środowisko naturalne dla grafów (friend-of-a-friend).

### Kiedy graf naprawdę "błyszczy" w systemach Agentowych?

Na podstawie doświadczeń z produkcji, graf sprawdza się w specyficznych warstwach pamięci agenta:

1.  Modelowanie wiedzy pojęciowej (knowledge graph). Koncept A (np. Python) jest wymagany do Konceptu B (np. Django). Agent może wnioskować: "Jeśli użytkownik nie zna Pythona, nie proponuj mu Django".
2.  Wnioskowanie przez relacje (Reasoning). "Jeśli użytkownik jest ekspertem w X, a X jest mocno powiązane z Y, to prawdopodobnie zrozumie też Y.
3. Złożone hierarchie i taksonomie. Drzewa kategorii, struktury organizacyjne firm, gdzie "rodzic" i "dziecko" to kluczowe pojęcia.

### ⚠️ Gdzie inżynierowie łamią sobie zęby?

1. Mylenie atrybutów z relacjami. Początkujący często tworzą węzeł dla każdego adresu email i łączą go z użytkownikiem relacją `HAS_EMAIL`.
- ❌ Źle: Węzeł `User` -> Relacja `HAS_EMAIL` -> Węzeł `email@example.com`
- ✅ Dobrze: Adres email to po prostu pole (atrybut) w węźle `User`. Nie komplikuj.

2. Brak eksperta w zespole
Zły model w bazie SQL boli. Zły model w bazie grafowej **zabija projekt**. Debugowanie zapytań w języku Cypher (używanym w Neo4j), gdy model jest nieintuicyjny, jest o rząd wielkości trudniejsze niż w SQL.

3. Bariera Językowa (Cypher) – czyli o co chodzi z tymi strzałkami? `<--[]-->`

To jest punkt, o którym zapomina 99% managerów decydujących o technologii. Bazy grafowe używają języka **[Cypher](https://neo4j.com/docs/getting-started/cypher/)**.

Na slajdach wygląda "fajnie", bo rysujesz strzałki. Ale spójrzmy prawdzie w oczy: to specyficzna, niszowa umiejętność. Na rynku jest 1000 razy mniej ekspertów od Cyphera niż od SQL. Jeśli Twój problem wygląda na "grafowy", ale Twój zespół zna tylko SQL, to wdrożenie bazy grafowej może być biznesowym samobójstwem. Dlaczego?

Pokażę na przykładzie. Załóżmy, że zadanie brzmi:
> *"Znajdź mi ścieżkę dojścia do decydenta, ale tylko przez osoby zaufane (omijając czarną listę), gdzie po drodze ktoś pracował w tej samej firmie, co nasza konkurencja."*

Brzmi jak konkretne zadanie logiczne, prawda?
A teraz spójrz na kod i powiedz mi szczerze: czy rozumiesz, co tu się dzieje?

#### Rozwiązanie (Cypher):

```cypher
MATCH path = (me:Person {id: $myId})-[:FRIEND|WORKED_WITH*1..4]-(middleman)
             -[:MANAGES]->(:Department)<-[:CONTRACTED_BY]-(target:Company)
WHERE NOT (me)-[:BLOCKED]-(middleman)
  AND none(n IN nodes(path) WHERE n.status IN ['BANNED', 'SUSPICIOUS'])
  AND single(x IN nodes(path) WHERE (x)-[:USED_TO_WORK]->(:Company {name: 'CompetitorX'}))
WITH me, target, path, relationships(path) as rels
MATCH (target)<-[:INVESTED_IN]-(vc:VC)-[:BOARD_MEMBER]->(final_boss:Person)
WHERE size([r IN rels WHERE type(r) = 'FRIEND']) > 1
RETURN me.name, final_boss.name, [n IN nodes(path) | n.lastname] as chain
```

Jejku... co to za dziwne znaczki?! 🤯

Widzisz to?
- `<-[:CONTRACTED_BY]-` — strzałki w lewo, w prawo, łańcuchy węzłów.
- `[:FRIEND|WORKED_WITH*1..4]` — "znajdź relację taką LUB taką, ale w odległości od 1 do 4 skoków".
- `none(...)`, `single(...)` — funkcje operujące na listach węzłów wewnątrz ścieżki.
- Logika `WITH` — która działa jak "rura" przekazująca dane do kolejnej części zapytania (pipeline).


W SQL patrzysz na kod i mniej więcej wiesz, co się dzieje. Tutaj? Jeden zły zwrot strzałki (`<-` zamiast `->`) albo pomyłka w głębokości ścieżki (`*1..5`) i Twoje zapytanie zamiast trwać 10ms, mieli procesor przez godzinę, próbując przejść przez cały internet.

Jeśli nie masz w zespole kogoś, kto *czuje* te strzałki, to nawet idealny use-case grafowy skończy się fiaskiem. Czasem lepiej napisać trzy brzydkie `JOIN-y` w SQL, które każdy junior zrozumie i naprawi, niż stworzyć taką "elegancką" czarną skrzynkę.

Baza grafowa to potężne działo. Nie używaj go do zabijania komarów (prostych CRUD-ów). Używaj go tam, gdzie relacje są bardziej skomplikowane niż same dane.

## Porównanie baz

Teraz praktyczne porównanie konkretnych technologii:

### [Neo4j](https://neo4j.com)

```
✅ Najlepszy gdy:
- Naprawdę modelujesz relacje między encjami
- Potrzebujesz zapytań "przez wiele hopów" 
  (znajomi znajomych znajomych)
- Knowledge graph z ontologią
- Dependency analysis (co od czego zależy)

⚠️ Uważaj na:
- ACID jest, ale transakcje między wieloma node'ami kosztowne
- Cypher (język zapytań) ma krzywą uczenia
- Monitoring i debugowanie trudniejsze niż SQL
- Skalowanie horyzontalne drogie (Enterprise only)

Sizing reality check:
- <100k nodes: Świetnie działa
- 100k-10M nodes: Trzeba myśleć o indexach
- >10M nodes: Potrzebujesz eksperta

Dla pamięci agenta:
→ Semantic Memory (relacje między konceptami)
→ User knowledge graph ("user zna X, X jest częścią Y")
```

### [Redis](https://redis.io)

```
✅ Najlepszy gdy:
- Potrzebujesz ultra-niskiej latencji (<1ms)
- Cache + szybka sesja
- Dane efemeryczne (TTL)
- Pub/Sub między komponentami
- Rate limiting

⚠️ Uważaj na:
- Dane w RAM = kosztowne przy dużej objętości
- Persistence (AOF/RDB) ma trade-offy
- Cluster mode dodaje complexity
- Redis Stack (wektory, grafy) to inna bestia niż vanilla Redis

Sizing reality check:
- Ile RAM potrzebujesz ≈ ile danych chcesz trzymać
- 1M kluczy po 1KB = ~1GB RAM minimum
- W praktyce 2-3x więcej (fragmentacja)

Dla pamięci agenta:
→ Working Memory (primary!)
→ Session state
→ Hot cache dla często używanych danych
```

### [Redis Stack](https://redis.io/about/redis-stack/) (z wektorami i grafem)

```
⚠️ Kontrowersyjna opinia:

Redis Stack próbuje być "wszystkim dla wszystkich".
W praktyce oznacza to kompromisy wszędzie.

Wektory w Redis:
- Dobre dla małych zbiorów (<100k wektorów)
- Prostsze niż osobna baza wektorowa
- ALE: brak zaawansowanych features Qdrant/Pinecone

Graf w Redis (RedisGraph):
- Cypher-compatible
- Szybki dla prostych zapytań
- ALE: słabszy niż Neo4j dla złożonych grafów
- UWAGA: RedisGraph został wycofany! (2023)

Rekomendacja:
Używaj Redis do tego, w czym jest najlepszy (cache, sesje).
Dla wektorów/grafów → dedykowane bazy.
```

### [Chroma](https://www.trychroma.com)

```
✅ Najlepszy gdy:
- Prototypujesz / MVP
- Lokalny development
- Mały dataset (<100k dokumentów)
- Chcesz prostoty (Python native)

⚠️ Uważaj na:
- NIE jest production-ready dla dużej skali
- Persistence może być problematyczna
- Brak zaawansowanego monitoringu
- Limited query capabilities vs Qdrant

Sizing reality check:
- Świetny do 50k wektorów
- 50k-200k: zaczyna zwalniać
- >200k: rozważ migrację do Qdrant/Weaviate

Dla pamięci agenta:
→ Development & testing
→ Małe projekty / PoC
→ Embedded scenarios
```

### [Qdrant](https://qdrant.tech)

```
✅ Najlepszy gdy:
- Produkcja z wymaganiami na skalowalność
- Potrzebujesz zaawansowanego filtrowania
- Multi-tenancy (wielu użytkowników/agentów)
- Hybrid search (wektory + filtry)

⚠️ Uważaj na:
- Overhead operacyjny (kolejna baza do utrzymania)
- Potrzebujesz przemyślanej strategii indeksowania
- Collection design ma duży wpływ na performance

Mocne strony:
- Filtering jest NAPRAWDĘ dobry
- Quantization (mniejsze wektory = mniej RAM)
- Good documentation
- Active development

Dla pamięci agenta:
→ Semantic Memory (document search)
→ Episodic Memory (similarity search)
→ Gdy Chroma nie wystarcza
```

### [Weaviate](https://weaviate.io)

```
✅ Najlepszy gdy:
- Chcesz multi-modal (tekst + obrazy)
- GraphQL API jest preferowane
- Potrzebujesz wbudowanych modeli embeddingowych
- Hybrid search out-of-the-box

⚠️ Uważaj na:
- Cięższy niż Qdrant (więcej zasobów)
- GraphQL może być overkill dla prostych use-case'ów
- Schema management bardziej złożone

Porównanie z Qdrant:
- Weaviate: Więcej możliwości, większa złożoność
- Qdrant: Bardziej focused, lżejszy

Dla pamięci agenta:
→ Gdy potrzebujesz multi-modal
→ Gdy zespół zna GraphQL
→ Gdy chcesz "batteries included"
```

## Wzorzec adaptera (*persistence layer*) 

Teraz przejdźmy do kodu, który pokazałem na końcu lekcji 2.5 (zerknij do folderu `src`). To, co tam widzisz, to nie przypadek – to fundament architektury Software 3.0.

### Złota zasada: Twój kod logiczny (Agent) nigdy nie powinien wiedzieć, jakiej bazy danych używasz.

Jeśli w środku logiki agenta napiszesz na sztywno `chromadb.Client(...)` albo `psycopg2.connect(...)`, to właśnie "zepsułeś" swój projekt. Zamykasz sobie drogę do łatwego skalowania i testowania.

Zamiast tego stosujemy **dependency inversion**. W Twoim projekcie wygląda to tak:

### Kontrakt (interfejs)
Najpierw definiujemy, co nasza pamięć *musi umieć robić*, nie martwiąc się *jak* to robi.

```python
from abc import ABC, abstractmethod
from typing import Optional, Dict, Any

class PersistenceBackend(ABC):
    """
    Abstrakcyjny kontrakt dla dowolnego systemu zapisu.
    Nie obchodzi nas czy to RAM, plik, Redis czy Postgres.
    """
    
    @abstractmethod
    def save(self, user_id: str, state: Dict[str, Any]) -> None:
        pass

    @abstractmethod
    def load(self, user_id: str) -> Optional[Dict[str, Any]]:
        pass
        
    @abstractmethod
    def exists(self, user_id: str) -> bool:
        pass
```

### Domyślny wybór: In-Memory (startuj lekko)
Zwróć uwagę na plik `src/persistence/in_memory.py`. To jest nasza domyślna implementacja.

Dlaczego to jest genialne w swojej prostocie?
- Zero setupu. Nie musisz instalować Dockera, stawiać Postgresa ani konfigurować Redisa, żeby zacząć pisać logikę.
- Szybkość. Testy przelatują w milisekundach.
- Czystość. Resetujesz aplikację i masz czystą kartę.

To idealne rozwiązanie na start. Dopóki prototypujesz, trzymanie stanu w zmiennej (słowniku) jest absolutnie wystarczające.

### Magia konfiguracji 
A co, gdy wchodzimy na produkcję i mamy miliony rekordów? Wtedy (i tylko wtedy!) zmieniamy bazę. Ale w kodzie agenta nie zmieniamy **ani jednej linijki**.

Spójrz na `src/config.py`:

```python
class Config:
    # ... inne klucze ...

    # Persistence configuration
    # Tu dzieje się magia. Zmieniasz jedną wartość stringa:
    PERSISTENCE_BACKEND = "in_memory"  # Opcje: "in_memory", "redis", "postgres"
    
    REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")
```

W fabryce (factory) aplikacji mamy prostego `if-a`:
- Jeśli `backend == "in_memory"` → użyj klasy `InMemoryBackend`.
- Jeśli `backend == "redis"` → użyj klasy `RedisBackend`.
- Jeśli `backend == "postgres"` → użyj klasy `SQLBackend`.

Dla Agenta to bez różnicy. On po prostu woła `.save()`.

### Inżynierski pro-tip lub granulacja adapterów
To podejście ma jeszcze jedną, ukrytą zaletę. Mamy różne rodzaje warstwach pamięci, prawda?

Dzięki adapterom możesz mieszać technologie w ramach jednego systemu.

### Tabela mapowania warstwa → baza:

| Warstwa | Primary Storage | Secondary | Anty-pattern |
|---------|----------------|-----------|--------------|
| Constitutional | Git (pliki) | - | Baza wektorowa |
| Working | Redis | SQL (backup) | Każda persystentna jako primary |
| Workflow | SQL | Redis (duża skala) | Graf, wektory |
| Episodic | SQL | Qdrant (similarity search) | Tylko wektory |
| Semantic | SQL + Qdrant | Neo4j (dla relacji) | Tylko jeden typ |
| Procedural | Git + SQL | - | Wektory do "discovery" |


Wszystko to spina się w jednym systemie, sterowane prostą konfiguracją. To jest właśnie dojrzała inżynieria, a nie sklejanie kodu "na ślinę".

## Checklista inżyniera - zanim wybierzesz bazę

W świecie inżynierii decyzje oparte na "wydaje mi się" to prosta droga do długu technologicznego. Aby uniknąć paraliżu decyzyjnego, przygotowałem dla Ciebie konkretną checklistę.

Wypełnij ją szczerze przed startem projektu.


### 1. Wzorzec dostępu (*access pattern*)
```markdown
↳ Czy szukam głównie po ID/Kluczu? (Key-Value)
↳ Czy szukam zakresami (daty, ceny)? (Range)
↳ Czy szukam po znaczeniu/podobieństwie? (Similarity)
↳ Czy szukam po głębokich relacjach? (Graph traversal)
```

### 2. Spójność i trwałość (*consistency*) 
```markdown
↳ Czy muszę mieć gwarancję ACID? (np. transakcje finansowe)
↳ Czy potrzebuję błyskawicznego dostępu (<1ms)?
↳ Czy dane są "gorące" (stan rozmowy) czy "zimne" (archiwum faktur)?
```

### 3. Skala 
```markdown
↳ Mała: < 1 mln rekordów (mieści się w RAM)
↳ Średnia: 1 mln - 100 mln rekordów (standardowa baza relacyjna)
↳ Ogromna: > 100 mln rekordów (wymaga shardingu / Big Data)
```

### 4. Zespół
```markdown
↳ Czy mam w zespole eksperta od tej konkretnej technologii?
↳ Jeśli baza padnie w nocy, kto ją podniesie?
```


## Jak interpretować wyniki?

Odpowiedzi na powyższe pytania układają się w konkretne profile. Nie musisz zgadywać. 

Popatrzmy na przykłady.

### Scenariusz A: "Potrzebuję faktów i spokoju" (Long-term Memory)
Twoje odpowiedzi: Szukasz po ID lub zakresach, wymagasz spójności (ACID), skala jest mała/średnia.
Wtedy najlepszym wyborem będzie SQL, np. 🏆 PostgreSQL. To najbezpieczniejszy wybór dla **pamięci faktograficznej**. Jeśli potrzebujesz trochę wektorów, dodajesz `pgvector`. Jeśli potrzebujesz trochę JSON-a, używasz `JSONB`. Nie wymyślaj koła na nowo dla danych, które mają przetrwać lata.


### Scenariusz B: "Potrzebuję Pamięci Roboczej" (Working Memory) 🧠
Twoje odpowiedzi: Wzorzec dostępu to proste `Key-Value`, potrzebujesz szybkości (<1ms), przechowujesz bieżący stan agenta i historię rozmowy. To wtedy pewnie chcesz użyć **Redis**.

> **⚠️ Stop! Obalamy mit:**
> Być może myślisz: *"Zaraz, Redis to tylko cache, jak zrestartuję serwer, to dane znikną"*.
>
> **To bzdura.**
> W profesjonalnym środowisku (np. Google Cloud Memorystore, AWS ElastiCache) Redis jest *bazą trwałą (Persistent)*. Dane są replikowane i zrzucane na dysk.
>
> Redis jest idealny dla agentów nie tylko dlatego, że jest szybki, ale dlatego, że jego struktury danych mapują się na mózg agenta:
> - **Listy:** Idealne do trzymania historii czatu (łatwe przycinanie okna kontekstu)
> - **Hashmapy:** Idealne do trzymania stanu (profil użytkownika, bieżący krok)
> - **TTL:** Automatyczne zapominanie wątków (np. po 24h)
>
> Traktuj Redisa jako **trwałą, szybką pamięć operacyjną**, a nie tylko tymczasowy śmietnik.

### Scenariusz C: "Eksploruję nieznane" (*semantic search*)
Twoje odpowiedzi: Głównym (i jedynym) celem jest wyszukiwanie podobieństwa w milionach dokumentów. Fakty i relacje są drugorzędne. Wtedy warto postawić na bazę wektorową np. Qdrant / Weaviate / Chroma. Do tego warto patrzeć na dedykowane bazy wektorowe, bo mają lepsze algorytmy kompresji (HNSW) i skalowania dla samych wektorów niż Postgres (`pgvector`) czy Redis Stack (ma wsparcie wektorów). Warto w nie wejść, gdy Twoim głównym produktem jest wyszukiwarka semantyczna.

### Scenariusz D: "Detektyw" (*deep relations*)
Twoje odpowiedzi wskazują na pytania typu "kto zna kogoś, kto...", skala jest duża, a **masz w zespole eksperta od grafów**. To wtedy Neo4j/FalkorDB. Tylko tutaj zysk z wydajności zapytań grafowych przewyższy ogromny koszt utrzymania nowej technologii i bariery wejścia (Cypher).



### Podsumowanie w jednym diagramie


![](../images/select_db.jpg)


## Największe błędy, których unikać

### Błąd #1: "Wszystko do wektorów"

```
Symptom:
"Mamy 500k dokumentów w Pinecone i search działa wolno/źle"

Diagnoza:
- Wektory zajmują dużo miejsca (1536 floats × 4 bytes = 6KB/wektor)
- Bez metadanych nie można filtrować
- Bez filtrowania wszystko jest "podobne"

Rozwiązanie:
1. Dodaj agresywne filtry PRZED vector search
2. Rozważ czy naprawdę potrzebujesz semantic search
3. Często keyword search + reranking LLM-em działa lepiej
```

### Błąd #2: "Graf na wszystko"

```
Symptom:
"Nasze zapytania do Neo4j trwają 30 sekund"

Diagnoza:
- Graf bez odpowiednich indeksów
- Zapytania skanujące cały graf
- Modelowanie "wszystko jest relacją"

Rozwiązanie:
1. Profile queries (EXPLAIN w Cypher)
2. Dodaj indeksy na properties używane w WHERE
3. Przemyśl czy to naprawdę powinien być graf
```

### Błąd #3: "Redis jako primary database"

```
Symptom:
"Straciliśmy dane sesji po restarcie serwera"

Diagnoza:
- Redis persistence źle skonfigurowane
- Lub wcale nie skonfigurowane
- Lub AOF/RDB nie nadąża za zapisami

Rozwiązanie:
1. Redis = cache + efemeryczne dane
2. Cokolwiek ważnego = też w persystentnej bazie
3. Skonfiguruj AOF jeśli musisz mieć persistence
```

### Błąd #4: "Jeden embedding model do wszystkiego"

```
Symptom:
"Wektory działają świetnie na dokumentację, ale źle na kod"

Diagnoza:
- Embedding model trenowany na tekście
- Kod ma inną semantykę
- "Podobny wektor" ≠ "rozwiązuje ten sam problem"

Rozwiązanie:
1. Różne kolekcje dla różnych typów contentu
2. Różne modele embeddingowe (code vs text)
3. Metadata filtering żeby szukać w odpowiednim zbiorze
```

## Laboratorium - dotknijmy metalu

Możemy godzinami rysować diagramy architektury, ale dopiero gdy uruchomisz kod, kropki w głowie zaczynają się (inaczej) łączyć.

W tej sekcji przejdziemy przez przegląd technologii. Nie chodzi o to, żebyś w 5 minut stał się ekspertem od każdej z nich. Chcę Ci pokazać **namacalną różnicę** w pracy z każdą z tych baz. Zobaczysz, że każda "czuje się" inaczej i służy do czegoś innego.

### ⚠️ Ważne założenie techniczne
Domyślnie na Twoim komputerze te bazy nie istnieją. Przy każdym przykładzie podam Ci polecenie (zaczynające się od `!`), które w środowisku Jupyter spróbuje uruchomić usługę, lub komentarz, jak zrobić to w terminalu.

Zaczynamy od przygotowania bibliotek.

## Wczytujemy biblioteki

Zgodnie z dobrą praktyką inżynierską, wszystkie importy trzymamy w jednym miejscu. Uruchom tę komórkę raz.


In [ ]:
import json
import time
import numpy as np

import redis
from redis.commands.search.field import VectorField, TextField
from redis.commands.search.query import Query

import chromadb
from qdrant_client import QdrantClient
from qdrant_client.models import PointStruct, Distance, VectorParams, Filter, FieldCondition, MatchValue

## `Redis`


Redis to fenomen. Zaczynał jako prosty cache, a dziś (dzięki **Redis Stack**) próbuje robić wszystko. Sprawdźmy jego oblicza w kontekście pamięci agenta.

Co prawda z modułu grafowego (**RedisGraph**) firma oficjalnie zrezygnowała w połowie 2023 roku ([źródło](https://redis.io/blog/redisgraph-eol/)). I to bardzo dobra wiadomość. To dowód na odzyskanie inżynierskiego "fokusu" – zamiast próbować być przeciętnym we wszystkim, Redis wraca do bycia mistrzem w szybkości i (teraz) wektorach.


Najpierw go uruchomimy. Uruchom komórkę poniżej i poczekaj, aż zobaczysz:
```
🚀 Starting Redis...
✅ Started.
```

In [ ]:
!redis-start

To oznacza, że masz dostęp do Redis i możesz się z nim połączyć — super! To teraz zróbmy coś praktycznego.

### A. Redis jako Key-Value (Idealny do: Working Memory ⚡)
Najprostszy, najszybszy. Używamy go do trzymania ID sesji i krótkiego bufora rozmowy. Tutaj Redis jest bezkonkurencyjny.

In [ ]:
r = redis.Redis(host='localhost', port=6379, decode_responses=True)
r.ping() # Sprawdźmy czy żyje

Jak widzisz, `True` oznacza, że udało się połączyć, więc idziemy dalej.

In [ ]:
session_id = "session_user_123"
working_memory = {
    "current_intent": "booking_table",
    "slot_verified": False,
    "last_user_message": "Chcę stolik na 19:00"
}

### Zapisujemy jako JSON 
Ustawiamy `TTL` na 10s (ustawmy krótki czas, aby sprawdzić) - samo się posprząta! To jest killer-feature dla agentów: zapominanie niepotrzebnych wątków.

In [ ]:
r.set(session_id, json.dumps(working_memory, ensure_ascii=False), ex=10)

### Odczyt

In [ ]:
data = r.get(session_id)
print(f"⚡ Working Memory (Redis KV): {data}")

Wynik operacji po pierwszym uruchomieniu:
```
⚡ Working Memory (Redis KV): {"current_intent": "booking_table", "slot_verified": false, "last_user_message": "Chcę stolik na 19:00"}
```

A teraz **odczekaj 10 sekund** i spróbuj odczytać to samo polecenie ponownie. Co widzisz?

```markdown
⚡ Working Memory (Redis KV): None
```

Widzisz co się stało? To jest **TTL (Time To Live)** w akcji. Dane same się "posprzątały".

To kluczowa cecha pamięci roboczej. W systemach produkcyjnych ustawiamy ten czas zazwyczaj **24 godziny** lub dłużej (w zależności od długości sesji). Tutaj ustawiłem 10 sekund celowo, żeby natychmiast zobaczyć mechanizm działania. Dzięki temu Twój Redis nie zamieni się w śmietnik starych, nieaktualnych wątków.

### B. Redis jako Vector DB (może pasować do *Short-term Semantic Memory*)
Mało kto wie, że Redis też może radzić sobie z wektorami. Gdy chcesz mieć "pamięć podręczną" dla semantyki (np. cache'owanie zapytań do LLM, żeby nie płacić dwa razy za to samo pytanie).


#### Definicja schematu (Index)
Redis wymaga jawnego utworzenia indeksu dla wektorów

In [ ]:
r.ft("idx:memories").create_index([
    TextField("content"),
    VectorField("vector", "FLAT", {"TYPE": "FLOAT32", "DIM": 4, "DISTANCE_METRIC": "COSINE"})
])

### Zapis (symulujemy wektor 4-wymiarowy dla uproszczenia)
W produkcji tu byłby embedding z OpenAI, Gemini itd...

In [ ]:
dummy_vector = np.array([0.1, 0.9, 0.2, 0.5], dtype=np.float32).tobytes()

r.hset("memory:1", mapping={
    "content": "The user likes Italian cuisine",
    "vector": dummy_vector
})

### Wyszukiwanie (KNN)
KNN to matematyczna definicja problemu (znalezienie "k" najbliższych punktów), ale w bazach wektorowych realizuje się ją na dwa główne sposoby: dokładny (*Exact KNN*), który jest wolniejszy, oraz przybliżony (*ANN - Approximate Nearest Neighbors*), który stawia na szybkość kosztem minimalnej utraty precyzji.

Oto najpopularniejsze algorytmy indeksowania (sposoby realizacji szukania):

1.  **FLAT (Brute Force)** – Sprawdza każdy wektor z każdym. Daje 100% dokładności (Exact KNN), ale jest bardzo wolny przy milionach rekordów.
2.  **HNSW (Hierarchical Navigable Small World)** – Buduje wielowarstwowy graf połączeń między wektorami. **Obecny standard rynkowy** dla wyszukiwania w pamięci RAM. Jest niesamowicie szybki i dość dokładny (ANN).
3.  **IVF (Inverted File Index)** – Dzieli przestrzeń na klastry i przeszukuje tylko te najbardziej obiecujące. Popularny w starszych systemach (np. FAISS), dobry do ograniczania zużycia pamięci.

#### Co wspiera Redis?
Redis (poprzez moduł RediSearch/Vector) wspiera natywnie dwa najważniejsze algorytmy:
1.  **FLAT** – gdy masz mało danych i zależy Ci na idealnej precyzji
2.  **HNSW** – gdy masz dużo danych i zależy Ci na maksymalnej szybkości (to jest domyślny wybór w większości wdrożeń Redisa)

Zbudujmy podobny wektor np. `[0.1, 0.9, 0.2, 0.4]`, przypomnę, że oryginalny był `[0.1, 0.9, 0.2, 0.5]` (różni się tylko ostatnia wartość 0.4 vs 0.5).

In [ ]:
query_vector = np.array([0.1, 0.9, 0.2, 0.4], dtype=np.float32).tobytes()

### Składnia zapytań Redisa jest specyficzna 🤦

To polecenie tworzy zapytanie do wyszukiwarki wektorowej, które znajduje najbardziej podobny element (KNN 1) do dostarczonego wektora vec. Wynikiem ma być treść (content) oraz liczbowy wynik podobieństwa (score), czyli jak bardzo pasuje do zapytania. Z kolei `dialect(2)` oznacza, że zapytanie ma zostać wykonane w wersji składni numer 2 — to po prostu tryb, w którym silnik wyszukiwarki interpretuje polecenie. W praktyce mówi to systemowi: "użyj wersji 2 języka zapytań", aby poprawnie odczytać konstrukcje takie jak KNN.

In [ ]:
q = Query("*=>[KNN 1 @vector $vec AS score]").return_fields("content", "score").dialect(2)

res = r.ft("idx:memories").search(q, query_params={"vec": query_vector})
    
if res.docs:
    print(f"⚡ Semantic Cache (Redis Vector): {res.docs[0].content} (Score: {res.docs[0].score})")
else:
    print("Brak wyników.")

Wynik `score` to liczba opisująca jak bardzo dany rekord jest podobny do zapytania — jej znaczenie zależy od zastosowanej metryki (np. im mniejsza, tym bliżej, albo im większa, tym bliżej). Możliwe wartości to zazwyczaj zakres od 0 do 1 lub dowolne liczby dodatnie, ale konkretne znaczenie zależy od ustawień wyszukiwarki i sposobu liczenia podobieństwa.


### ☝️ Zwróć uwagę
Wynik bliski 0 lub 1 nie musi oznaczać jednoznacznego dopasowania lub całkowitej różnicy. W wektorowych wyszukiwarkach sam absolutny `score` nie ma dużej wartości interpretacyjnej — liczy się kolejność wyników, a nie to, czy wynik jest bliski zeru lub jedynce. To oznacza, że nawet bardzo małe różnice w score mogą decydować o podobieństwie, a same liczby nie są intuicyjne bez znajomości metryki. No właśnie! Często są mylące i nie należy ich interpretować dosłownie, wielu inżynierów "sparzyło się" na wektorach, próbując nadać tym liczbom znaczenie większe, niż faktycznie mają.


Dobra, to zatrzymajmy redis i potestujemy chromadb.

In [ ]:
!redis-stop

## `Chroma` 🦆

ChromaDB to ulubieniec developerów. Działa lokalnie, nie wymaga serwera, instaluje się jednym poleceniem. Jest idealna do *Episodic Memory* w fazie developmentu.

Najpierw trzeba ją uruchomić (i chwilę poczekać).

In [ ]:
!chroma-start

O ile widzisz `✅ chroma started.`, to wtedy idziemy dalej.

### Klient 

In [ ]:
chroma_client = chromadb.Client()

### Kolekcja (odpowiednik tabeli)

In [ ]:
collection = chroma_client.create_collection(name="episodic_memory")

### Zapis 
Chroma ma "baterie w zestawie" - sama zrobi embeddingi domyślnym modelem, jeśli nie podasz własnych! To ogromne ułatwienie na start.

In [ ]:
collection.add(
    documents=["Użytkownik wspomniał, że ma alergie na orzechy", "Użytkownik szuka stolika w centrum"],
    metadatas=[{"type": "health"}, {"type": "preference"}],
    ids=["evt_1", "evt_2"]
)

## Wyszukiwanie semantyczne – jak czytać wynik?

W `ChromaDB` (i większości baz wektorowych) parametr `distance` działa odwrotnie niż intuicyjna "procentowa pewność". Traktuj to jak wynik w golfie – im mniej punktów, tym lepiej.

Wartość ta reprezentuje fizyczną odległość między dwoma punktami w wielowymiarowej przestrzeni:
- **0.0** – tożsame (ten sam tekst).
- **0.3 - 0.6** – Bliskie znaczeniowo (np. "pies" i "szczeniak").
- **> 1.0** – Dalekie, mało wspólnego (zależnie od modelu).

Chroma domyślnie używa **Squared L2** (kwadrat odległości euklidesowej). Jeśli więc szukasz "centrum miasta", a wynik ma `dist: 1.45`, to znaczy, że baza "dopycha" wynik na siłę, bo nie znalazła nic bliskiego. Przy `L2` wynik może być większy niż 1.0, w przeciwieństwie do *Cosine Similarity*, które zwykle zamyka się w 0-1 (lub 0-2 w zależności od implementacji dystansu).

### Co napędza ChromaDB? (silnik i algorytmy)

ChromaDB różni się od Redisa tym, że jest dedykowaną, "natywną" bazą wektorową. Nie oferuje wielu algorytmów do wyboru – stawia na jedną, sprawdzoną i zoptymalizowaną ścieżkę.
1.  **HNSW (Implementacja `hnswlib`)** – To serce Chromy. Baza wykorzystuje popularną bibliotekę C++ (`hnswlib`), która realizuje algorytm przybliżonego sąsiedztwa. *Dlaczego to ważne?* Dzięki temu Chroma jest bardzo szybka nawet przy milionach rekordów, ale zużywa pamięć RAM do przechowywania grafu połączeń.
2.  **Brak trybu FLAT (domyślnie)** – W przeciwieństwie do Redisa, Chroma nie pyta Cię, czy chcesz używać *Brute Force*. Zawsze buduje indeks HNSW, aby zapewnić skalowalność, nawet jeśli na początku masz mało danych (choć przy mikro-zestawach danych działa to de facto jak brute force).

#### Dostępne metryki dystansu (*distance metrics*)
To tutaj masz największą kontrolę w Chromie. Przy zakładaniu kolekcji (`create_collection`) definiujesz parametr `metadata={"hnsw:space": "..."}`:

1.  **`l2` (Squared L2)** – [Domyślna]. Najbardziej surowa. Premiuje wektory, które są blisko siebie w przestrzeni euklidesowej. Świetna, gdy zależy Ci na precyzyjnym dopasowaniu.
2.  **`cosine` (Cosine Similarity)** – Ignoruje długość wektora, skupia się na kącie między nimi. **Standard w NLP** (przetwarzaniu tekstu), ponieważ lepiej radzi sobie z dopasowywaniem tematów (np. długi dokument o psach vs. krótkie zdanie o psie).
3.  **`ip` (Inner Product)** – Iloczyn skalarny. Używany w specyficznych modelach rekomendacyjnych (nieznormalizowanych).

Jest to rozwiązanie typu "battery included". Domyślnie dostajesz **HNSW + L2**, co jest bezpiecznym startem, ale dla lepszych wyników w języku naturalnym (tekstach) warto rozważyć przy tworzeniu kolekcji aby wymusić metrykę *Cosine*.

In [ ]:
results = collection.query(
    query_texts=["Czy są jakieś ograniczenia dietetyczne?"],
    n_results=5
)

for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f"Score:{dist:.4f} -> {doc}")

Jak widzisz, suche liczby bywają zwodnicze: wektory mogą mieć bardzo zbliżony score, sugerujący podobieństwo, mimo że w rzeczywistości dotyczą zupełnie odrębnych tematów. To właśnie główna pułapka pracy z wektorami – patrząc na wynik końcowy, musimy decydować między opcją A a B, gdzie minimalna różnica liczbowa może przesądzać o trafności odpowiedzi. Dlatego samo „szukanie podobnych” nie wystarczy; aby uniknąć błędów przy tak małych różnicach, konieczne jest użycie modelu, który rozumie niuanse językowe, wyraźniej oddziela kluczowe informacje od szumu i wymaga jeszcze wielu eksperymentów – mierzenia, czy faktycznie wszystko działa wystarczająco dobrze, aby zaakceptować wynik w produkcji ("good enough").


No to zatrzymajmy Chroma i przejdźmy do Qdrant.

In [ ]:
!chroma-stop

## `Qdrant`

Gdy wchodzisz na produkcję z baza wektorowa, możesz potrzebować *Qdranta*. Jego supermocą jest filtrowanie. W pamięci agenta to kluczowe – chcesz szukać "wspomnień o pizzy", ale TYLKO z "ostatniego tygodnia".


In [ ]:
!qdrant-start

O ile widzisz `✅ qdrant started.`, to możemy zacząć używać. Najpierw inicjalizacja klienta.
### Klient

In [ ]:
q_client = QdrantClient(host="localhost", port=6333)

## Tworzymy kolekcję 

Musimy zdefiniować rozmiar wektora, np. 4 dla demo.

In [ ]:
q_client.recreate_collection(
    collection_name="semantic_knowledge",
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
)

### Zapis (Punkt = Wektor + Payload)

In [ ]:
q_client.upsert(
    collection_name="semantic_knowledge",
    points=[
        PointStruct(id=1, vector=[0.1, 0.9, 0.2, 0.1], payload={"topic": "food", "user": "Agata"}),
        PointStruct(id=2, vector=[0.1, 0.9, 0.2, 0.1], payload={"topic": "food", "user": "Adam"}),
    ]
)

### Co napędza Qdrant? 

`Qdrant` to technologiczny "muscle car" napisany w języku **Rust**, co zapewnia mu ogromną wydajność i stabilność przy zarządzaniu zasobami. W odróżnieniu od ogólnych baz danych z "doklejonym" modułem wektorowym, Qdrant od początku był projektowany do jednego celu: **wyszukiwania z filtrowaniem**.

Oto co znajduje się pod maską:

1.  **HNSW (Custom Implementation)** – Qdrant również używa standardu rynkowego HNSW, ale w **zmodyfikowanej, autorskiej wersji**. *Dlaczego to ważne?* Zwykły HNSW gubi się, gdy każesz mu szukać i filtrować jednocześnie (tzw. problem *pre-filtering* vs *post-filtering*). Qdrant zmodyfikował algorytm tak, aby filtr (np. `user_id`) był sprawdzany **w trakcie** poruszania się po grafie wektorów.
2.  **Osobne indeksy dla Payloadu** – Qdrant traktuje metadane (payload) jako "obywateli pierwszej kategorii". Tworzy dla nich dedykowane struktury (np. drzewa B-Tree, GeoHash), co sprawia, że złożone zapytania typu *"znajdź podobne wektory, ale tylko z wczoraj, w promieniu 5km i dla użytkownika X"* są błyskawiczne.

#### Dostępne metryki dystansu (Distance Metrics)
Przy tworzeniu kolekcji w Qdrant musisz zadeklarować, jak chcesz mierzyć "podobieństwo". Ma to kluczowy wpływ na interpretację wyniku:

1.  **Cosine Similarity (`Cosine`)** – [Najpopularniejsza]. Mierzy kąt między wektorami. Idealna do tekstów i NLP (podobnie jak w OpenAI).
2.  **Dot Product (`Dot`)** – Iloczyn skalarny. Używany, gdy wektory są znormalizowane lub gdy długość wektora niesie informację (np. w systemach rekomendacji).
3.  **Euclidean Distance (`Euclid`)** – Fizyczna odległość. Używana rzadziej w NLP, częściej w przetwarzaniu obrazu.



### Jak czytać wynik w Qdrant?

Tutaj uwaga – Qdrant domyślnie posługuje się pojęciem **`score` (wynik punktowy)**, a nie `distance` (odległość), chyba że używasz metryki Euklidesowej.

Zależy to od wybranej metryki:

*   **Dla Cosine / Dot Product (Standard):**
    *   Im **WYŻSZY** wynik, tym **LEPIEJ**.
    *   **1.0** = Identyczne (to samo znaczenie).
    *   **0.7 - 0.9** = Bardzo wysokie podobieństwo.
    *   **< 0.5** = Słabe powiązanie (zależnie od modelu).
    *   *Zasada:* Szukamy najwyższych słupków.

*   **Dla Euclidean:**
    *   Im **NIŻSZY** wynik, tym **LEPIEJ** (jak w ChromaDB).
    *   **0.0** = Identyczne.
    *   *Zasada:* Szukamy wartości najbliższych zera.

W Qdrant z metryką Cosine (najczęstszy przypadek dla tekstów) wynik **0.92** oznacza świetne dopasowanie, podczas gdy w Chroma (używającej domyślnie L2) wynik **0.92** mógłby oznaczać już spory dystans. Zawsze sprawdzaj konfigurację kolekcji!

In [ ]:
hits = q_client.query_points(
    collection_name="semantic_knowledge",
    query=[0.1, 0.9, 0.2, 0.1], 
    query_filter=Filter(              
        must=[
            FieldCondition(
                key="user", 
                match=MatchValue(value="Agata")
            )
        ]
    ),
    limit=1
).points  


if hits:
    print(f"🦀 Qdrant (Filtered): Znaleziono ID: {hits[0].id} (Payload: {hits[0].payload})")
else:
    print("🦀 Qdrant: Nic nie znaleziono.")

Oto podsumowanie Qdranta i przejście do kolejnego gracza.

### Podsumowanie Qdrant
Qdrant to wydajnościowa bestia, która dzięki świetnemu silnikowi filtrującemu daje ogromne możliwości precyzyjnego wyszukiwania, ale trzeba to powiedzieć wprost: jego składnia bywa „barokowa” i specyficzna. Co więcej, jak właśnie zauważyłeś na żywym organizmie, twórcy potrafią dość agresywnie zmieniać nazwy metod między wersjami (np. znikające `search`), co bywa źródłem sporej frustracji przy utrzymywaniu kodu.

Zatrzymujmy `qdrant` (aby zwolnić zasoby).

In [ ]:
!qdrant-stop

## Podsumowanie

To była długa, ale konieczna podróż przez fundamenty inżynierii danych w AI. Zamiast ślepo wierzyć w "magiczną moc wektorów", spojrzeliśmy pod maskę i sprawdziliśmy, jak te systemy działają na poziomie metalu.

Oto co konkretnie dowieźliśmy w tej lekcji:

- **Obaliliśmy mit "Vector-Only"** – wiesz już, dlaczego pytanie o cenę produktu zadaną bazie wektorowej kończy się katastrofą i dlaczego SQL nadal jest królem faktów.
- **Zrozumieliśmy pułapkę grafową** – masz jasność, że modelowanie wszystkiego jako węzłów i relacji (Neo4j) bez wyraźnej potrzeby to prosty sposób na "zabicie" projektu skomplikowanym utrzymaniem.
- **Dotknęliśmy kodu** – uruchomienie Redisa pokazało, jak działa TTL (samooczyszczanie pamięci), a porównanie Chromy i Qdranta uświadomiło różnicę między prostym prototypem a produkcyjnym filtrowaniem.
- **Pokazałem wzorzec Adaptera** – twój agent jest teraz uniezależniony od bazy danych. Zmiana z "in-memory" na "Qdrant" to teraz kwestia jednej linijki w konfiguracji, a nie przepisywania całej aplikacji.
- **Stworzyliśmy mapę drogową** – wiesz, która technologia pasuje do Pamięci Roboczej (Redis), a która do Semantycznej (Qdrant/Weaviate).



### Najważniejsze wnioski

**Abstrakcja ratuje życie.** Twój kod agenta nie powinien zależeć bezpośrednio od `chromadb` czy `redis-py`. Buduj interfejsy (Adaptery). Dzięki temu zmiana bazy w przyszłości będzie kwestią konfiguracji, a nie przepisywania systemu.

#### Specjalizacja pamięci:
- Redis → Prędkość (Working Memory)
- Wektory → Semantyka (Episodic Memory)
- Git/Pliki → Zasady i Instrukcje (Constitutional Memory)
- SQL → Twarde Fakty (ceny, daty, logi)

> **Złota zasada:** "Jeśli nie potrafisz wyjaśnić juniorowi w 2 minuty, dlaczego wybrałeś tę konkretną bazę dla tej warstwy pamięci, prawdopodobnie wybrałeś źle."

> "Najlepsza architektura to ta, którą możesz łatwo zmienić, gdy okaże się zła."



## Co dalej?

Mamy solidny fundament. Twój agent ma już "mózg", który potrafi zapamiętywać kontekst, fakty i znaczenie, a co najważniejsze – nie traci ich po restarcie serwera. Ale na razie jest to mózg zamknięty w słoiku. Może myśleć, ale nie może działać.

W kolejnym module zdejmiemy te kajdany. Zajmiemy się **Tool Use (Function Calling)**.

Nauczymy agenta wychodzić poza okno czatu – łączyć się z API, przeszukiwać internet i wykonywać operacje na plikach. 

Wszystko zaczyna się łączyć w całość...

## Przydatne linki

1. [Agentic Memory](https://neo4j.com/videos/agentic-memory/)
2. [Relational to Graph Data Modeling | Michael Hunger, Developer Relations at Neo4j](https://www.youtube.com/watch?v=Hs4O17kjQXA)
3. [Redis Stack Documentation](https://redis.io/docs/stack/)
4. [Qdrant: Filtering is all you need](https://qdrant.tech/)
5. [Neo4j for Graph RAG](https://neo4j.com/generativeai/)
6. [PostgreSQL pgvector](https://github.com/pgvector/pgvector)
7. [Building AI Agents with Knowledge Graph Memory: A Comprehensive Guide to Graphiti](https://medium.com/@saeedhajebi/building-ai-agents-with-knowledge-graph-memory-a-comprehensive-guide-to-graphiti-3b77e6084dec)